In [ ]:
# Setup packages and paths
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import os
import time
import matplotlib.colors as mcolors
from pyscenic.aucell import aucell
import gseapy as gp
from gseapy import SingleSampleGSEA
from collections import namedtuple
from scipy import sparse

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RESULT_DIR = PROJECT_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)


In [ ]:
# Load filtered AnnData
file_path = DATA_DIR / "GSE201333_RAW" / "GSM6058681_TabulaSapiens.h5ad" / "adata_filtered.h5ad"
adata = sc.read_h5ad(str(file_path))

print(adata.shape)
print(adata)


In [ ]:
# Inspect tissue and tissue-celltype annotations
print(adata.shape)
print(adata.obs['organ_tissue'].value_counts())
print(adata.obs['tissue_celltype'].value_counts())


In [ ]:
# Identify tissue-specific marker genes
sc.tl.rank_genes_groups(
    adata,
    groupby='organ_tissue',
    method='wilcoxon',
    n_genes=adata.shape[1],
    pts=True,
    use_raw=False
)

marker_lists = []

for tissue in adata.obs['organ_tissue'].cat.categories:
    print(f"Processing cell type: {tissue} …")
    start = time.time()
    
    df = sc.get.rank_genes_groups_df(adata, group=tissue)
    
    pct_cols = [
        col for col in df.columns
        if col.lower().startswith('pct') and 'group' in col.lower()
    ]
    if not pct_cols:
        raise KeyError(
            "Cannot find a column indicating expression proportion in group; "
            f"available columns: {df.columns.tolist()}"
        )
    pct_col = pct_cols[0]
    
    df_filt = df[
        (df['logfoldchanges'] >= 0.25) &
        (df[pct_col] >= 0.25)
    ]
    
    top100 = (
        df_filt
        .sort_values('logfoldchanges', ascending=False)
        .head(100)['names']
        .tolist()
    )
    marker_lists.extend(top100)
    
    elapsed = time.time() - start
    print(f" → {tissue}: selected {len(top100)} markers in {elapsed:.1f}s\n")

unique_markers = list(dict.fromkeys(marker_lists))

pd.Series(unique_markers).to_csv(
    RESULT_DIR / 'tissue_marker_top100.txt',
    index=False,
    header=False
)
print(f"Total unique markers saved: {len(unique_markers)}")


In [ ]:
# Export AnnData and metadata for R conversion
obs = adata.obs.copy()
for col in obs.select_dtypes(include=["category", "object"]).columns:
    obs[col] = obs[col].astype(str)

var = pd.DataFrame({"gene_symbol": adata.var_names}, index=adata.var_names)
X = adata.X.copy()

layers = {}
if "raw_counts" in adata.layers:
    layers["raw_counts"] = adata.layers["raw_counts"].copy()

want_obsm = {k: adata.obsm[k].copy()
             for k in ["X_pca", "X_umap", "X_scvi"]
             if k in adata.obsm}

adata_sub = sc.AnnData(
    X=X,
    obs=obs,
    var=var,
    layers=layers,
    obsm=want_obsm
)

adata_sub.uns.clear()
adata_sub.obsp.clear()
adata_sub.varm.clear()
adata_sub.varp.clear()

obs.to_csv(
    RESULT_DIR / "adata_obs_metadata.csv",
    index=True,
    index_label="cell_id",
    encoding="utf-8"
)
print("Saved: adata_obs_metadata.csv")

adata_sub.write_h5ad(RESULT_DIR / "filtered_subset_for_convert.h5ad")
print("Saved: filtered_subset_for_convert.h5ad")
print(adata_sub)


In [ ]:
# Export raw count matrix as Feather
raw = adata.layers['raw_counts']
sparse.issparse(raw)
dense_counts = raw.toarray()

df_counts = pd.DataFrame(
    dense_counts,
    index=adata.obs_names,
    columns=adata.var_names
)

df_counts.to_feather(RESULT_DIR / "filtered_sc_counts_Deconvolution.feather")


In [ ]:
# Subset key tissues for downstream celltype integration
print(f"Original data shape: {adata.shape}")
print(f"Unique organ_tissue values: {adata.obs['organ_tissue'].unique()}")

target_tissues = ["Vasculature", "Blood", "Heart", "Skin"]

mask = adata.obs['organ_tissue'].isin(target_tissues)
subset_adata = adata[mask, :]

print(f"Original cell number: {adata.n_obs}")
print(f"Filtered cell number: {subset_adata.n_obs}")
print(f"Selected tissue types: {target_tissues}")

output_filename = RESULT_DIR / "key_4tissue_celltype_adata.h5ad"
subset_adata.write_h5ad(output_filename)
print(f"Saved: {output_filename}")


In [ ]:
# Clear user-defined variables
for name in dir():
    if not name.startswith('_'):
        del globals()[name]
